In [ ]:
# import solcx

# # install compiler
# solcx.install_solc("0.4.26")
# solcx.install_solc("0.5.17")

# # set version
# solcx.set_solc_version("0.4.26")

# source_code = """
# pragma solidity ^0.4.26;

# contract Test {
#     function foo() public pure returns (uint) {
#         return 1;
#     }
# }
# """


# compiled = solcx.compile_source(source_code)

In [ ]:
# import re
# import os
# import solcx
# import pandas as pd
# import numpy as np
# import subprocess
# import tempfile
# import pickle
# from tqdm import tqdm

# # =========================
# # PRAGMA
# # =========================
# def extract_pragma(code):
#     match = re.search(r'pragma solidity([^;]+);', code)
#     return match.group(1).strip() if match else None

# def normalize_version(v):
#     if v is None:
#         return "0.4.26"

#     v = v.replace(" ", "")

#     if "0.4" in v:
#         return "0.4.26"
#     elif "0.5" in v:
#         return "0.5.17"

#     return "0.4.26"

# # =========================
# # SOLC SETUP
# # =========================
# installed_versions = set()

# def ensure_solc(version):
#     global installed_versions

#     if version in installed_versions:
#         solcx.set_solc_version(version)
#         return

#     try:
#         solcx.set_solc_version(version)
#     except:
#         solcx.install_solc(version)
#         solcx.set_solc_version(version)

#     installed_versions.add(version)

# # =========================
# # RUN SLITHER
# # =========================
# def run_slither_cfg(code, version):
#     ensure_solc(version)

#     if "pragma solidity" not in code:
#         code = f"pragma solidity ^{version};\n" + code

#     with tempfile.NamedTemporaryFile(delete=False, suffix=".sol", mode="w", encoding="utf-8") as f:
#         f.write(code)
#         file_path = f.name

#     try:
#         cmd = f"slither {file_path} --print cfg"
#         result = subprocess.run(
#             cmd,
#             shell=True,
#             capture_output=True,
#             text=True,
#             timeout=30
#         )

#         if result.returncode != 0:
#             return None

#         return result.stdout

#     except:
#         return None

#     finally:
#         os.remove(file_path)

# # =========================
# # PARSE CFG → ADJ
# # =========================
# def parse_cfg(output):
#     nodes = set()
#     edges = []

#     for line in output.split("\n"):
#         match = re.findall(r'Node (\d+)', line)
#         if len(match) == 2:
#             u, v = int(match[0]), int(match[1])
#             nodes.add(u)
#             nodes.add(v)
#             edges.append((u, v))

#     if len(nodes) == 0:
#         return [], [], np.zeros((1, 1), dtype=np.float32)

#     nodes = sorted(nodes)
#     node_map = {n: i for i, n in enumerate(nodes)}

#     mapped_edges = [(node_map[u], node_map[v]) for u, v in edges]

#     n = len(nodes)
#     adj = np.zeros((n, n), dtype=np.float32)

#     for u, v in mapped_edges:
#         adj[u][v] = 1.0

#     return nodes, mapped_edges, adj

# # =========================
# # EXTRACT NODE TEXT
# # =========================
# def extract_node_texts(cfg_output):
#     node_texts = {}

#     for line in cfg_output.split("\n"):
#         match = re.match(r'Node (\d+):\s*(.*)', line)
#         if match:
#             idx = int(match.group(1))
#             text = match.group(2).lower()
#             node_texts[idx] = text

#     return node_texts

# # =========================
# # FEATURE (STRONG VERSION)
# # =========================
# def build_node_features(cfg_output, nodes, adj):
#     node_texts = extract_node_texts(cfg_output)

#     n = len(nodes)

#     in_deg = adj.sum(axis=0)
#     out_deg = adj.sum(axis=1)

#     features = []

#     for i, original_id in enumerate(nodes):
#         text = node_texts.get(original_id, "")

#         # ===== STRUCTURAL =====
#         f_in = in_deg[i]
#         f_out = out_deg[i]
#         f_branch = 1 if f_out > 1 else 0
#         f_entry = 1 if f_in == 0 else 0
#         f_exit = 1 if f_out == 0 else 0

#         # ===== TOKEN =====
#         tokens = re.findall(r'\w+', text)
#         num_tokens = len(tokens)
#         unique_tokens = len(set(tokens))

#         # ===== SECURITY =====
#         has_call = int("call" in text)
#         has_delegatecall = int("delegatecall" in text)
#         has_transfer = int("transfer" in text)
#         has_send = int("send" in text)

#         has_msg_sender = int("msg.sender" in text)
#         has_msg_value = int("msg.value" in text)
#         has_tx_origin = int("tx.origin" in text)

#         has_require = int("require" in text)
#         has_assert = int("assert" in text)
#         has_revert = int("revert" in text)

#         has_selfdestruct = int("selfdestruct" in text or "suicide" in text)

#         # ===== CONTROL =====
#         has_if = int("if" in text)
#         has_loop = int("for" in text or "while" in text)

#         # ===== ARITH =====
#         num_arith = len(re.findall(r'[\+\-\*/%]', text))
#         num_compare = len(re.findall(r'(==|!=|>|<|>=|<=)', text))

#         # ===== STORAGE =====
#         has_state_write = int(bool(re.search(r'\w+\s*=\s*(?!=)', text)))

#         # ===== REENTRANCY =====
#         has_reentrancy_pattern = int(
#             ("call" in text or "send" in text or "transfer" in text)
#             and "=" in text
#         )

#         feat = [
#             f_in, f_out, f_branch, f_entry, f_exit,
#             num_tokens, unique_tokens,
#             has_call, has_delegatecall, has_transfer, has_send,
#             has_msg_sender, has_msg_value, has_tx_origin,
#             has_require, has_assert, has_revert,
#             has_selfdestruct,
#             has_if, has_loop,
#             num_arith, num_compare,
#             has_state_write,
#             has_reentrancy_pattern
#         ]

#         features.append(feat)

#     return np.array(features, dtype=np.float32)

# # =========================
# # MAIN
# # =========================
# def build_dataset(csv_path):
#     df = pd.read_csv(csv_path)

#     label_columns = [
#         'Other', 'access_control', 'arithmetic', 'denial_service',
#         'front_running', 'reentrancy', 'time_manipulation',
#         'unchecked_low_calls'
#     ]
#     labels = (df[label_columns].values > 0).astype(int)

#     feature_list = []
#     adj_list = []
#     node_counts = []

#     for _, row in tqdm(df.iterrows(), total=len(df)):
#         code = row["sourcecode"]

#         if not isinstance(code, str) or len(code) < 20:
#             adj = np.zeros((1, 1), dtype=np.float32)
#             features = np.zeros((1, 24), dtype=np.float32)
#         else:
#             pragma = extract_pragma(code)
#             version = normalize_version(pragma)

#             cfg_output = run_slither_cfg(code, version)

#             if cfg_output is None:
#                 adj = np.zeros((1, 1), dtype=np.float32)
#                 features = np.zeros((1, 24), dtype=np.float32)
#             else:
#                 nodes, edges, adj = parse_cfg(cfg_output)
#                 features = build_node_features(cfg_output, nodes, adj)

#         feature_list.append(features)
#         adj_list.append(adj)
#         node_counts.append(features.shape[0])

#     node_counts = np.array(node_counts)

#     print("Mean nodes:", node_counts.mean())
#     print("Min nodes:", node_counts.min())
#     print("Max nodes:", node_counts.max())

#     data = {
#         "features": feature_list,
#         "adj_matrices": adj_list,
#         "labels": labels
#     }

#     return data

# # =========================
# # RUN + SAVE
# # =========================
# if __name__ == "__main__":
#     data = build_dataset("DATASET/multilabel_BILSTM_BERT.csv")

#     with open("DATASET/gnn_input_slither.pkl", "wb") as f:
#         pickle.dump(data, f)

#     print("✅ Saved!")

  1%|          | 285/45597 [06:38<17:45:07,  1.41s/it]

In [19]:
import re
import os
import solcx
import pandas as pd
import numpy as np
import subprocess
import tempfile
import pickle
from tqdm import tqdm
from multiprocessing.dummy import Pool
from packaging import version

# =========================
# PRAGMA
# =========================
def extract_pragma(code):
    match = re.search(r'pragma solidity\s*([^;]+);', code)
    return match.group(1).strip() if match else None


def resolve_solc_version(pragma):
    if pragma is None:
        return "0.8.20"

    # exact version
    exact = re.search(r'\d+\.\d+\.\d+', pragma)
    if exact and "^" not in pragma and "<" not in pragma and ">" not in pragma:
        return exact.group(0)

    # major.minor
    m = re.search(r'(\d+\.\d+)', pragma)
    if not m:
        return "0.8.20"

    major_minor = m.group(1)

    available = sorted(
        solcx.get_installable_solc_versions(),
        key=lambda v: version.parse(v),
        reverse=True
    )

    for v in available:
        if v.startswith(major_minor):
            return v

    return "0.8.20"


# =========================
# SOLC SETUP
# =========================
def preload_solc():
    versions = ["0.4.26", "0.5.17", "0.6.12", "0.7.6", "0.8.20"]
    for v in versions:
        try:
            solcx.install_solc(v)
        except:
            pass


def get_solc_path(version):
    try:
        solcx.install_solc(version)
    except:
        pass

    return solcx.get_executable(version)


# =========================
# RUN SLITHER
# =========================
def run_slither_cfg(code, version):
    if "pragma solidity" not in code:
        code = f"pragma solidity {version};\n" + code

    solc_path = get_solc_path(version)

    with tempfile.NamedTemporaryFile(delete=False, suffix=".sol", mode="w", encoding="utf-8") as f:
        f.write(code)
        file_path = f.name

    try:
        cmd = [
            "slither",
            file_path,
            "--print", "cfg",
            "--solc", solc_path
        ]

        result = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            timeout=30
        )

        if result.returncode != 0:
            print("❌ Slither failed")
            print("Version:", version)
            print("Solc path:", solc_path)
            print("STDERR:\n", result.stderr)
            return None

        return result.stdout

    except subprocess.TimeoutExpired:
        print("⏰ TIMEOUT")
        return None

    except Exception as e:
        print("💥 EXCEPTION:", e)
        return None

    finally:
        try:
            os.remove(file_path)
        except:
            pass


# =========================
# PARSE CFG
# =========================
def parse_cfg(output):
    nodes = set()
    edges = []

    for line in output.split("\n"):
        match = re.findall(r'Node (\d+)', line)
        if len(match) == 2:
            u, v = int(match[0]), int(match[1])
            nodes.add(u)
            nodes.add(v)
            edges.append((u, v))

    if not nodes:
        return [], [], np.zeros((1, 1), dtype=np.float32)

    nodes = sorted(nodes)
    node_map = {n: i for i, n in enumerate(nodes)}

    mapped_edges = [(node_map[u], node_map[v]) for u, v in edges]

    n = len(nodes)
    adj = np.zeros((n, n), dtype=np.float32)

    for u, v in mapped_edges:
        adj[u][v] = 1.0

    return nodes, mapped_edges, adj


# =========================
# NODE TEXT
# =========================
def extract_node_texts(cfg_output):
    node_texts = {}

    for line in cfg_output.split("\n"):
        match = re.match(r'Node (\d+):\s*(.*)', line)
        if match:
            node_texts[int(match.group(1))] = match.group(2).lower()

    return node_texts


# =========================
# FEATURE
# =========================
def build_node_features(cfg_output, nodes, adj):
    node_texts = extract_node_texts(cfg_output)

    in_deg = adj.sum(axis=0)
    out_deg = adj.sum(axis=1)

    features = []

    for i, original_id in enumerate(nodes):
        text = node_texts.get(original_id, "")

        f_in = in_deg[i]
        f_out = out_deg[i]
        f_branch = 1 if f_out > 1 else 0
        f_entry = 1 if f_in == 0 else 0
        f_exit = 1 if f_out == 0 else 0

        tokens = re.findall(r'\w+', text)

        feat = [
            f_in, f_out, f_branch, f_entry, f_exit,
            len(tokens), len(set(tokens)),
            int("call" in text), int("delegatecall" in text),
            int("transfer" in text), int("send" in text),
            int("msg.sender" in text), int("msg.value" in text), int("tx.origin" in text),
            int("require" in text), int("assert" in text), int("revert" in text),
            int("selfdestruct" in text or "suicide" in text),
            int("if" in text), int("for" in text or "while" in text),
            len(re.findall(r'[\+\-\*/%]', text)),
            len(re.findall(r'(==|!=|>|<|>=|<=)', text)),
            int(bool(re.search(r'\w+\s*=\s*(?!=)', text))),
            int(("call" in text or "send" in text or "transfer" in text) and "=" in text)
        ]

        features.append(feat)

    return np.array(features, dtype=np.float32)


# =========================
# PROCESS ONE
# =========================
def process_one(row):
    code = row["sourcecode"]

    if isinstance(code, str):
        code = code.replace('\r\n', '\n').replace('\r', '\n')

    if not isinstance(code, str) or len(code.strip()) < 20:
        return np.zeros((1, 24), dtype=np.float32), np.zeros((1, 1), dtype=np.float32)

    pragma = extract_pragma(code)
    version = resolve_solc_version(pragma)

    cfg_output = run_slither_cfg(code, version)

    if cfg_output is None:
        return np.zeros((1, 24), dtype=np.float32), np.zeros((1, 1), dtype=np.float32)

    nodes, edges, adj = parse_cfg(cfg_output)
    features = build_node_features(cfg_output, nodes, adj)

    return features, adj


# =========================
# MAIN DATASET BUILD
# =========================
def build_dataset(csv_path):
    df = pd.read_csv(csv_path)
    df = df.head(1000)

    label_columns = [
        'Other', 'access_control', 'arithmetic', 'denial_service',
        'front_running', 'reentrancy', 'time_manipulation', 'unchecked_low_calls'
    ]

    labels = (df[label_columns].values > 0).astype(int)
    records = df.to_dict("records")

    with Pool(8) as p:
        results = list(tqdm(
            p.imap_unordered(process_one, records),
            total=len(records),
            desc="🚀 Processing contracts",
            unit="contract"
        ))

    feature_list = [r[0] for r in results]
    adj_list = [r[1] for r in results]

    node_counts = np.array([f.shape[0] for f in feature_list])

    print("Mean nodes:", node_counts.mean())
    print("Min nodes:", node_counts.min())
    print("Max nodes:", node_counts.max())

    return {
        "features": feature_list,
        "adj_matrices": adj_list,
        "labels": labels
    }


# =========================
# RUN
# =========================
if __name__ == "__main__":
    preload_solc()

    data = build_dataset("DATASET/multilabel_BILSTM_BERT.csv")

    with open("DATASET/gnn_input_slither.pkl", "wb") as f:
        pickle.dump(data, f)

    print("✅ Saved!")

🚀 Processing contracts:   0%|          | 0/1000 [00:00<?, ?contract/s]


AttributeError: module 'solcx' has no attribute 'get_executable'

In [15]:
sum

0

In [ ]:
import torch
import pickle
import numpy as np
import torch.nn.functional as F

from torch.nn import Linear, BatchNorm1d
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GATConv, global_mean_pool
from sklearn.metrics import precision_score, recall_score, f1_score

# =========================
# LOAD DATA
# =========================
input_file = 'DATASET/gnn_input_slither.pkl'

with open(input_file, 'rb') as f:
    raw_data = pickle.load(f)

feature_list = raw_data['features']
adj_matrices = raw_data['adj_matrices']
label_list = raw_data['labels']

target_dim = 24
num_classes = len(label_list[0])

def pad_or_truncate_features(features, target_dim):
    if features.shape[1] > target_dim:
        return features[:, :target_dim]
    elif features.shape[1] < target_dim:
        padded_features = np.zeros((features.shape[0], target_dim), dtype=np.float32)
        padded_features[:, :features.shape[1]] = features
        return padded_features
    else:
        return features.astype(np.float32)

def create_pyg_data(features, adj_matrix, label, target_dim):
    adj = torch.tensor(np.array(adj_matrix), dtype=torch.long)
    edge_index = adj.nonzero(as_tuple=False).t().contiguous()

    x = torch.tensor(pad_or_truncate_features(features, target_dim), dtype=torch.float)
    y = torch.tensor(label, dtype=torch.float)

    return Data(x=x, edge_index=edge_index, y=y)

data_list = [
    create_pyg_data(feat, adj, label, target_dim)
    for feat, adj, label in zip(feature_list, adj_matrices, label_list)
]

# nếu muốn shuffle trước khi split thì mở 2 dòng dưới
# indices = np.random.permutation(len(data_list))
# data_list = [data_list[i] for i in indices]

split_idx = int(len(data_list) * 0.8)
train_loader = DataLoader(data_list[:split_idx], batch_size=32, shuffle=True)
test_loader = DataLoader(data_list[split_idx:], batch_size=32, shuffle=False)


c:\Users\Admin\Desktop\ai_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


KeyboardInterrupt: 

In [ ]:

# =========================
# MODEL
# =========================
# class GAT(torch.nn.Module):
#     def __init__(self, in_channels, hidden_channels, out_channels, heads=4):
#         super(GAT, self).__init__()
#         self.conv1 = GATConv(in_channels, hidden_channels, heads=heads, dropout=0.3)
#         self.bn1 = BatchNorm1d(hidden_channels * heads)

#         self.conv2 = GATConv(hidden_channels * heads, hidden_channels, heads=1, concat=False, dropout=0.3)
#         self.bn2 = BatchNorm1d(hidden_channels)

#         self.lin = Linear(hidden_channels, out_channels)

#     def forward(self, data):
#         x, edge_index, batch = data.x, data.edge_index, data.batch

#         x = self.conv1(x, edge_index)
#         x = self.bn1(x)
#         x = F.relu(x)

#         x = self.conv2(x, edge_index)
#         x = self.bn2(x)
#         x = F.relu(x)

#         x = global_mean_pool(x, batch)
#         x = F.dropout(x, p=0.3, training=self.training)

#         x = self.lin(x)
#         x = torch.sigmoid(x)   # giữ sigmoid như bạn muốn
#         return x

class GAT(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, num_classes=8):
        super().__init__()

        self.conv1 = GATConv(in_channels, hidden_channels, heads=4)
        self.conv2 = GATConv(hidden_channels * 4, hidden_channels, heads=4)
        self.conv3 = GATConv(hidden_channels * 4, hidden_channels)

        self.lin = torch.nn.Linear(hidden_channels, num_classes)

        self.dropout = 0.3

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch

        # Layer 1
        x = self.conv1(x, edge_index)
        x = F.elu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        # Layer 2
        x = self.conv2(x, edge_index)
        x = F.elu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        # Layer 3
        x = self.conv3(x, edge_index)

        # Pooling
        x = global_mean_pool(x, batch)
        x = F.dropout(x, p=0.3, training=self.training)

        x = self.lin(x)
        x = torch.sigmoid(x)   # giữ sigmoid như bạn muốn
        return x
    
# =========================
# SETUP
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device = torch.device("cpu")

gnn_model = GAT(in_channels=target_dim, hidden_channels=64).to(device)
criterion = torch.nn.BCELoss()
optimizer = torch.optim.Adam(gnn_model.parameters(), lr=1e-3, weight_decay=1e-4)

# =========================
# TRAIN / EVAL
# =========================
def train_one_epoch(model, loader, optimizer, criterion, device, threshold=0.5):
    model.train()
    total_loss = 0.0
    all_preds = []
    all_labels = []

    for data in loader:
        data = data.to(device)

        optimizer.zero_grad()
        out = model(data)   # out là xác suất vì đã sigmoid

        y = data.y
        if y.dim() == 1:
            y = y.view(-1, num_classes)

        loss = criterion(out, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        preds = (out >= threshold).float()

        all_preds.append(preds.detach().cpu().numpy())
        all_labels.append(y.detach().cpu().numpy())

    avg_loss = total_loss / len(loader)
    all_preds = np.concatenate(all_preds, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)

    precision = precision_score(all_labels, all_preds, average='micro', zero_division=0)
    recall = recall_score(all_labels, all_preds, average='micro', zero_division=0)
    f1 = f1_score(all_labels, all_preds, average='micro', zero_division=0)

    return avg_loss, precision, recall, f1

In [ ]:

# =========================
# TRAIN LOOP
# =========================
num_epochs = 30
best_test_loss = float("inf")

for epoch in range(num_epochs):
    train_loss, train_precision, train_recall, train_f1 = train_one_epoch(
        gnn_model, train_loader, optimizer, criterion, device
    )

    print(
        f"Epoch {epoch+1:02d}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Precision: {train_precision:.4f} | "
        f"Recall: {train_recall:.4f} | "
        f"F1: {train_f1:.4f}"
    )

    # nếu bạn muốn theo dõi luôn trên test mỗi epoch thì mở phần dưới
    # test_loss, test_precision, test_recall, test_f1, _, _ = evaluate(
    #     gnn_model, test_loader, criterion, device
    # )
    # print(
    #     f"                Test Loss: {test_loss:.4f} | "
    #     f"Precision: {test_precision:.4f} | "
    #     f"Recall: {test_recall:.4f} | "
    #     f"F1: {test_f1:.4f}"
    # )
